# Lezione 04 — Probabilità per l'analista

**Corso:** Statistica Applicata con Python · **Blocco 2:** Probabilità
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/04_probabilidad_analista_IT.md`

## Cosa vedrai in questa lezione

- La probabilità come **frequenza a lungo termine**, verificata con la simulazione.
- **Eventi**, spazio campionario, evento complementare, unione e intersezione.
- **Indipendenza** e **probabilità condizionata** (e perché P(A|B) non è P(B|A)).
- La **legge della probabilità totale** e il **teorema di Bayes** applicati a un test diagnostico.
- La **simulazione Monte Carlo** confrontata con il calcolo esatto.

## Prerequisiti

- Lezioni da 1 a 3 (variabilità campionaria, frequenze e distribuzioni empiriche).
- Librerie: `numpy`, `pandas`, `matplotlib` (`pip install numpy pandas matplotlib`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada; poi confrontalo con quello che vedi. Tutti i numeri del caso del test diagnostico sono un *esempio illustrativo*, non dati reali.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme** del generatore casuale: con lo stesso seme, tutti otteniamo gli stessi numeri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. La probabilità come frequenza a lungo termine

Lanciamo una moneta equilibrata 100 000 volte e, dopo ogni lancio, calcoliamo la **frequenza relativa** di teste. L'idea: la probabilità di un evento è la proporzione di volte in cui si verifica se ripeti l'esperimento moltissime volte.

In [ ]:
N_LANZ = 100_000
lanzamientos = rng.random(N_LANZ) < 0.5                     # True = testa
frec_acum = np.cumsum(lanzamientos) / np.arange(1, N_LANZ + 1)

filas = [{"lanci": n,
          "frequenza di teste": frec_acum[n - 1],
          "distanza da 0,5": abs(frec_acum[n - 1] - 0.5)}
         for n in [10, 100, 1_000, 10_000, 100_000]]
display(pd.DataFrame(filas).set_index("lanci").round(4))

# cinco trayectorias independientes de 2 000 lanzamientos
for _ in range(5):
    tray = np.cumsum(rng.random(2_000) < 0.5) / np.arange(1, 2_001)
    plt.plot(np.arange(1, 2_001), tray)
plt.axhline(0.5, color="black", linestyle="--")
plt.xscale("log")
plt.title("Frequenza relativa di teste: cinque esperimenti")
plt.xlabel("Numero di lanci (scala log)"); plt.ylabel("Frequenza di teste")
plt.show()

**Cosa osservi?**

- All'inizio (pochi lanci) le traiettorie si disperdono molto. Cosa succede alla dispersione al crescere del numero di lanci?
- Verso quale valore convergono tutte e cinque? Arrivano *esattamente* a 0,5?
- Cosa ti dice questo sul significato di "la probabilità di testa è 0,5" per un singolo lancio?

*(La tua risposta qui.)*

---
## 2. Spazio campionario ed eventi

Con un dado a sei facce, lo **spazio campionario** è Ω = {1, 2, 3, 4, 5, 6}. Definiamo due eventi: A = "esce pari" e B = "esce più di 4". Calcoliamo le probabilità **esatte** contando i casi (tutti i risultati sono equiprobabili) e le verifichiamo con una simulazione.

In [ ]:
omega = np.arange(1, 7)
A = omega[omega % 2 == 0]          # pari: {2, 4, 6}
B = omega[omega > 4]               # maggiore di 4: {5, 6}

def prob(evento):
    """Probabilità esatta contando i casi (risultati equiprobabili)."""
    return len(evento) / len(omega)

union = np.union1d(A, B)
interseccion = np.intersect1d(A, B)

print(f"P(A) = {prob(A):.4f}    P(B) = {prob(B):.4f}")
print(f"P(non A) = 1 − P(A)            = {1 - prob(A):.4f}")
print(f"P(A ∪ B) = {prob(union):.4f}    P(A ∩ B) = {prob(interseccion):.4f}")
print(f"Regola della somma: P(A) + P(B) − P(A ∩ B) = {prob(A) + prob(B) - prob(interseccion):.4f}")

# Simulazione: 200 000 lanci
tiradas = rng.integers(1, 7, size=200_000)
en_A, en_B = np.isin(tiradas, A), np.isin(tiradas, B)
print()
print(f"Simulazione: P(A) ≈ {en_A.mean():.4f}   P(B) ≈ {en_B.mean():.4f}   "
      f"P(A ∪ B) ≈ {(en_A | en_B).mean():.4f}   P(A ∩ B) ≈ {(en_A & en_B).mean():.4f}")

**Osserva:** la simulazione si avvicina ai valori esatti ma non li riproduce al centesimo. Nota che la regola della somma sottrae l'intersezione: senza quel termine conteremmo due volte il risultato 6, che appartiene sia ad A sia a B.

---
## 3. Indipendenza e fallacia del giocatore

Due eventi sono **indipendenti** se sapere che uno si è verificato non cambia la probabilità dell'altro. Con due dadi, "il primo è 6" e "il secondo è 6" lo sono, quindi P(entrambi sono 6) = 1/6 · 1/6 = 1/36.

In [ ]:
N = 500_000
d1 = rng.integers(1, 7, size=N)
d2 = rng.integers(1, 7, size=N)
ambos_6 = (d1 == 6) & (d2 == 6)

print(f"Esatto 1/36 = {1/36:.5f}   Simulazione = {ambos_6.mean():.5f}")
print(f"P(2° è 6 | 1° è 6) ≈ {(d2[d1 == 6] == 6).mean():.4f}     (1/6 = {1/6:.4f})")

### Esperimento guidato: dopo una serie di croci "tocca" testa?

Molte persone credono che, dopo cinque croci di fila, la testa sia "dovuta". Lo verifichiamo: lanciamo una moneta 1 000 000 di volte, cerchiamo tutte le posizioni precedute da cinque croci consecutive e calcoliamo la frequenza di testa nel lancio successivo.

In [ ]:
N = 1_000_000
caras = rng.random(N) < 0.5                       # True = testa
cruces = (~caras).astype(int)

c = np.concatenate(([0], np.cumsum(cruces)))      # c[k] = croci nei primi k lanci
racha_5 = (c[5:N] - c[0:N - 5]) == 5              # i 5 lanci precedenti sono stati croce
siguiente = caras[5:]

print(f"Frequenza globale di testa              : {caras.mean():.4f}")
print(f"Posizioni precedute da 5 croci       : {racha_5.sum():,}")
print(f"Frequenza di testa dopo una serie di 5 croci: {siguiente[racha_5].mean():.4f}")

**Cosa osservi?**

- La frequenza di testa dopo cinque croci è maggiore, minore o uguale a quella globale?
- Cosa implica questo sulla moneta che "ricorda" quello che è successo? Applicalo a un giocatore di roulette che punta sul rosso "perché sono già usciti cinque neri".

*(La tua risposta qui.)*

---
## 4. Quando NON c'è indipendenza: estrazione senza reinserimento

Se estrai due carte da un mazzo di 52 **senza rimettere** la prima, il secondo risultato dipende dal primo. Probabilità esatta che siano entrambe assi: 4/52 · 3/51. Moltiplicare 4/52 · 4/52 (come se fossero indipendenti) sarebbe un errore.

In [ ]:
N = 100_000
orden = rng.random((N, 52)).argsort(axis=1)[:, :2]      # due carte distinte per ogni mazzo simulato
ambos_ases = (orden < 4).all(axis=1)                    # le carte 0-3 sono gli assi

exacto = (4 / 52) * (3 / 51)
erroneo = (4 / 52) ** 2
print(f"Esatto (senza reinserimento) : {exacto:.5f}")
print(f"Errore: trattarli come indipendenti : {erroneo:.5f}")
print(f"Simulazione ({N:,} mazzi)      : {ambos_ases.mean():.5f}")

**Osserva:** la simulazione coincide con il calcolo esatto e scarta la "scorciatoia" sbagliata, che sovrastima la probabilità di circa il 30 %.

---
## 5. Probabilità condizionata con una tabella di contingenza

Un negozio registra 1 000 clienti secondo il canale di contatto e se hanno acquistato (*esempio illustrativo*, dati sintetici). La probabilità condizionata si calcola **restringendo la tabella** alla riga o colonna che ci interessa: P(A | B) = P(A ∩ B) / P(B).

In [ ]:
tabla = pd.DataFrame({"acquista": [120, 90], "non acquista": [280, 510]},
                     index=["web", "negozio"])
display(tabla)

total = tabla.values.sum()
n_web = tabla.iloc[0].sum()
n_compra = tabla.iloc[:, 0].sum()
n_web_compra = tabla.iloc[0, 0]

print(f"P(acquista)                    = {n_compra / total:.3f}")
print(f"P(acquista | web)              = {n_web_compra / n_web:.3f}")
print(f"P(acquista | negozio)           = {tabla.iloc[1, 0] / tabla.iloc[1].sum():.3f}")
print(f"P(web | acquista)              = {n_web_compra / n_compra:.3f}      ← non è P(acquista | web)!")

**Osserva:** P(acquista | web) = 0,30 e P(web | acquista) ≈ 0,57 sono cose diverse: la prima risponde a "tra chi arriva dal web, quanti acquistano?", la seconda a "tra chi acquista, quanti sono arrivati dal web?". Inoltre, poiché P(acquista | web) ≠ P(acquista), canale e acquisto **non sono indipendenti**.

---
## 6. Caso guida: un test diagnostico (frequenze naturali)

Un test per una malattia ha **sensibilità** 95 % (individua il 95 % dei malati), **specificità** 95 % (risulta negativo nel 95 % dei sani) e la **prevalenza** è dell'1 %. *(Tutti questi numeri sono un esempio illustrativo.)*

Prima di usare Bayes facciamo la cosa più intuitiva: immaginare **10 000 persone** e contare quante finiscono in ogni casella.

In [ ]:
PREVALENCIA = 0.01         # configurabile: esempio illustrativo
SENSIBILIDAD = 0.95
ESPECIFICIDAD = 0.95
N_PERSONAS = 10_000

def frecuencias_naturales(prevalencia, sensibilidad, especificidad, n=10_000):
    """Quante persone di ogni tipo ci sono su n."""
    enfermos = n * prevalencia
    sanos = n - enfermos
    vp = enfermos * sensibilidad            # veri positivi
    fn = enfermos - vp                      # falsi negativi
    fp = sanos * (1 - especificidad)        # falsi positivi
    vn = sanos - fp                         # veri negativi
    return {"enfermos": enfermos, "sanos": sanos, "vp": vp, "fn": fn, "fp": fp, "vn": vn}

def vpp_exacto(prevalencia, sensibilidad, especificidad):
    """Valore predittivo positivo: P(malato | positivo), con Bayes."""
    numerador = prevalencia * sensibilidad
    return numerador / (numerador + (1 - prevalencia) * (1 - especificidad))

def vpn_exacto(prevalencia, sensibilidad, especificidad):
    """Valore predittivo negativo: P(sano | negativo)."""
    numerador = (1 - prevalencia) * especificidad
    return numerador / (numerador + prevalencia * (1 - sensibilidad))

f = frecuencias_naturales(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD, N_PERSONAS)
tabla_nat = pd.DataFrame(
    {"Malati": [f["vp"], f["fn"], f["enfermos"]],
     "Sani": [f["fp"], f["vn"], f["sanos"]]},
    index=["Test positivo", "Test negativo", "Total"])
tabla_nat["Total"] = tabla_nat.sum(axis=1)
display(tabla_nat.round(0).astype(int))

print(f"Positivi totali: {f['vp'] + f['fp']:.0f}  →  di cui, malati davvero: {f['vp']:.0f}")
print(f"VPP = P(malato | positivo) = {vpp_exacto(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD):.4f}")
print(f"VPN = P(sano | negativo)     = {vpn_exacto(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD):.4f}")

plt.bar(["Test positivo"], [f["vp"]], color="seagreen", label="Veri positivi")
plt.bar(["Test positivo"], [f["fp"]], bottom=[f["vp"]], color="crimson", label="Falsi positivi")
plt.title("Chi risulta positivo? (10 000 persone, esempio illustrativo)")
plt.ylabel("Persone"); plt.legend()
plt.show()

**Cosa osservi?**

- Su 10 000 persone, quante risultano positive? Quante di loro sono davvero malate?
- Un test "preciso al 95 %" ha ragione il 95 % delle volte sui malati e sui sani, eppure la maggior parte dei positivi è falsa. Quale numero della tabella lo spiega?
- Se il tuo risultato è positivo, che probabilità hai di essere malato? Confrontala con la tua intuizione iniziale.

*(La tua risposta qui.)*

---
## 7. Bayes esatto contro simulazione Monte Carlo

Simuliamo un milione di persone: ognuna è malata con probabilità pari alla prevalenza e risulta positiva con probabilità pari alla sensibilità (se malata) o a 1 − specificità (se sana). Stimiamo P(malato | positivo) come la **frazione di malati tra i positivi** e la confrontiamo con il valore esatto. L'errore standard di una proporzione è √(p(1−p)/n), con n il numero di positivi.

In [ ]:
def simular_test(prevalencia, sensibilidad, especificidad, n, rng):
    """Simula n persone: restituisce (malato, positivo) come array booleani."""
    enfermo = rng.random(n) < prevalencia
    u = rng.random(n)
    positivo = np.where(enfermo, u < sensibilidad, u < 1 - especificidad)
    return enfermo, positivo

N_MC = 1_000_000
enfermo_mc, positivo_mc = simular_test(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD, N_MC, rng)

vpp_ex = vpp_exacto(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD)
n_pos = positivo_mc.sum()
vpp_sim = enfermo_mc[positivo_mc].mean()
ee = np.sqrt(vpp_ex * (1 - vpp_ex) / n_pos)

print(f"Positivi nella simulazione : {n_pos:,} de {N_MC:,}")
print(f"VPP esatto (Bayes)          : {vpp_ex:.4f}")
print(f"VPP simulato                 : {vpp_sim:.4f}   (errore standard ≈ {ee:.4f})")
print(f"Differenza in errori standard     : {abs(vpp_sim - vpp_ex) / ee:.2f}")
print()
print(f"P(positivo | malato) ≈ {positivo_mc[enfermo_mc].mean():.4f}   ← sensibilità")
print(f"P(malato | positivo) ≈ {vpp_sim:.4f}   ← sono cose molto diverse!")

**Cosa osservi?** A quanti errori standard si trova la simulazione dal valore esatto? Cosa ti dice l'ultima coppia di righe sul confondere P(A|B) con P(B|A)?

Ora ripetiamo la simulazione con un numero diverso di persone per vedere come migliora la stima.

In [ ]:
filas = []
for n in [1_000, 10_000, 100_000, 1_000_000]:
    enf, pos = simular_test(PREVALENCIA, SENSIBILIDAD, ESPECIFICIDAD, n, rng)
    est = enf[pos].mean()
    filas.append({"persone simulate": n, "positivi": int(pos.sum()),
                  "VPP simulato": est, "errore assoluto": abs(est - vpp_ex)})
display(pd.DataFrame(filas).set_index("persone simulate").round(4))

**Osserva:** con poche persone simulate ci sono pochissimi positivi e la stima oscilla; moltiplicando N per 10 l'errore scende all'incirca della radice di 10 (≈ 3 volte). È lo stesso fenomeno della lezione 1: anche la simulazione ha rumore campionario.

---
## 8. L'effetto della prevalenza

Cosa succede al valore predittivo positivo (VPP) se lo stesso test (sensibilità e specificità fisse) viene applicato a popolazioni con prevalenza diversa? Calcoliamo la curva con la formula esatta.

In [ ]:
prevalencias = np.linspace(0.001, 0.5, 500)
for esp in [0.90, 0.95, 0.99]:
    plt.plot(prevalencias, vpp_exacto(prevalencias, 0.95, esp), label=f"specificità {esp:.0%}")
plt.axvline(PREVALENCIA, color="gray", linestyle=":", label="prevalenza del caso guida (1 %)")
plt.xscale("log")
plt.title("VPP in funzione della prevalenza (sensibilità 95 %, esempio illustrativo)")
plt.xlabel("Prevalenza (scala log)"); plt.ylabel("VPP = P(malato | positivo)"); plt.legend()
plt.show()

filas = [{"prevalenza": p, "VPP": vpp_exacto(p, 0.95, 0.95),
          "falsi ogni 1 000 positivi": 1000 * (1 - vpp_exacto(p, 0.95, 0.95))}
         for p in [0.001, 0.01, 0.05, 0.10, 0.50]]
display(pd.DataFrame(filas).set_index("prevalenza").round(3))

prev_50 = (1 - 0.95) / (0.95 + 1 - 0.95)
print(f"Con sensibilità = specificità = 95 %, il VPP raggiunge il 50 % quando la prevalenza è {prev_50:.0%}.")

**Cosa osservi?**

- Cosa succede al VPP quando la malattia è molto rara? E quando la prevalenza è alta?
- Quanto aiuta migliorare la specificità dal 95 % al 99 % quando la prevalenza è bassa?
- Perché i test di screening si usano soprattutto su popolazioni a rischio e non sull'intera popolazione?

*(La tua risposta qui.)*

---
## 9. Un secondo test: la probabilità si aggiorna

Dopo un primo positivo, la probabilità di essere malato passa dalla prevalenza (1 %) al VPP (≈ 16 %). Se ripetiamo un test **indipendente** (dato lo stato di salute), quel 16 % diventa la nuova "prevalenza" di partenza. Lo calcoliamo con Bayes e lo verifichiamo sulle persone risultate positive nella simulazione precedente.

In [ ]:
vpp_dos_tests = vpp_exacto(vpp_ex, SENSIBILIDAD, ESPECIFICIDAD)      # la probabilità a posteriori diventa a priori

idx = np.flatnonzero(positivo_mc)                                    # quelli risultati positivi al primo test
u2 = rng.random(idx.size)
positivo_2 = np.where(enfermo_mc[idx], u2 < SENSIBILIDAD, u2 < 1 - ESPECIFICIDAD)
vpp_2_sim = enfermo_mc[idx][positivo_2].mean()

print(f"Dopo un positivo              : {vpp_ex:.4f}")
print(f"Dopo due positivi (esatto)     : {vpp_dos_tests:.4f}")
print(f"Dopo due positivi (simulazione) : {vpp_2_sim:.4f}")

**Osserva:** un secondo positivo porta la probabilità di essere malato a circa il 78 %. Ma attenzione: questo vale solo se gli errori dei due test sono indipendenti. Se uno stesso fattore (per esempio una sostanza che interferisce) provoca il falso positivo in entrambi, i risultati non sono indipendenti e il calcolo sovrastima l'effetto.

---
## Riepilogo della lezione

1. **Probabilità:** frequenza relativa a lungo termine; la simulazione converge al valore esatto, con un rumore che scende come 1/√n.
2. **Eventi:** il complementare somma 1 e la regola della somma sottrae l'intersezione.
3. **Indipendenza:** P(A ∩ B) = P(A)·P(B) solo se sono indipendenti; i lanci di una moneta non si "compensano".
4. **Condizionata:** P(A|B) = P(A ∩ B)/P(B), e **P(A|B) ≠ P(B|A)**.
5. **Bayes e frequenze naturali:** con prevalenza bassa, anche un test con sensibilità e specificità del 95 % dà una maggioranza di falsi positivi (VPP ≈ 16 % con prevalenza dell'1 %, esempio illustrativo).
6. **Monte Carlo:** permette di stimare probabilità simulando; la stima va accompagnata dal suo errore standard.

**Prossimo passo:** Lezione 5 — Distribuzioni di probabilità e teorema del limite centrale. E ora, esercitati: `notebooks_practica/clase04_practica_probabilidad_analista_IT.ipynb`.